# 🤖 Analyse et Comparaison des Modeles

Ce notebook compare les performances des 3 modeles : Prophet, XGBoost et LSTM.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Chargement des donnees
df = pd.read_csv('../data/processed/all_tickers_clean.csv', index_col=0, parse_dates=True)
tickers = df['Ticker'].unique().tolist()
print(f'Tickers disponibles: {tickers}')

## 1. Entrainement et evaluation des modeles

In [ ]:
from models.prophet_model import train_prophet
from models.xgboost_model import train_xgboost
from models.lstm_model import train_lstm

# Entrainer sur un ticker
ticker = 'AAPL'
print(f'=== Entrainement sur {ticker} ===')

print('\n--- Prophet ---')
r_prophet = train_prophet(df, ticker)
print(f"MAE: {r_prophet['metrics']['mae']:.2f}, MAPE: {r_prophet['metrics']['mape']:.2f}%")

print('\n--- XGBoost ---')
r_xgboost = train_xgboost(df, ticker)
print(f"MAE: {r_xgboost['metrics']['mae']:.2f}, MAPE: {r_xgboost['metrics']['mape']:.2f}%")

print('\n--- LSTM ---')
r_lstm = train_lstm(df, ticker, epochs=30)
print(f"MAE: {r_lstm['metrics']['mae']:.2f}, MAPE: {r_lstm['metrics']['mape']:.2f}%")

## 2. Comparaison des metriques sur tous les tickers

In [ ]:
# Entrainer tous les modeles sur tous les tickers
results = []

for ticker in tickers:
    print(f'Entrainement {ticker}...')
    
    r = train_prophet(df, ticker)
    if r:
        results.append({'Ticker': ticker, 'Modele': 'Prophet', 
                       'MAE': r['metrics']['mae'], 'MAPE': r['metrics']['mape']})
    
    r = train_xgboost(df, ticker)
    if r:
        results.append({'Ticker': ticker, 'Modele': 'XGBoost',
                       'MAE': r['metrics']['mae'], 'MAPE': r['metrics']['mape']})
    
    r = train_lstm(df, ticker, epochs=30)
    if r:
        results.append({'Ticker': ticker, 'Modele': 'LSTM',
                       'MAE': r['metrics']['mae'], 'MAPE': r['metrics']['mape']})

df_results = pd.DataFrame(results)
df_results.round(2)

In [ ]:
# Graphique comparatif MAPE
fig = px.bar(df_results, x='Ticker', y='MAPE', color='Modele',
             barmode='group', title='MAPE par modele et ticker (%)',
             text='MAPE')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=500)
fig.show()

In [ ]:
# Graphique comparatif MAE
fig = px.bar(df_results, x='Ticker', y='MAE', color='Modele',
             barmode='group', title='MAE par modele et ticker ($)',
             text='MAE')
fig.update_traces(texttemplate='$%{text:.1f}', textposition='outside')
fig.update_layout(height=500)
fig.show()

## 3. Meilleur modele par ticker

In [ ]:
# Quel modele est le meilleur pour chaque ticker ?
best = df_results.loc[df_results.groupby('Ticker')['MAPE'].idxmin()]
print('Meilleur modele par ticker (MAPE la plus basse):')
print(best[['Ticker', 'Modele', 'MAPE']].to_string(index=False))

## 4. Importance des features (XGBoost)

In [ ]:
# Feature importance pour chaque ticker
from models.xgboost_model import train_xgboost

for ticker in tickers[:3]:
    r = train_xgboost(df, ticker)
    if r:
        imp = pd.Series(r['feature_importance']).sort_values(ascending=True)
        fig = px.barh(x=imp.values, y=imp.index,
                      title=f'Importance des features — {ticker}')
        fig.update_layout(height=300)
        fig.show()

## 5. Previsions Prophet

In [ ]:
# Prevision a 30 jours
from models.prophet_model import train_prophet

ticker = 'AAPL'
r = train_prophet(df, ticker, forecast_days=30)

forecast = r['forecast']
hist = df[df['Ticker'] == ticker].tail(90)

fig = go.Figure()
fig.add_trace(go.Scatter(x=hist.index, y=hist['Close'], name='Historique', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=forecast['ds'].tail(30), y=forecast['yhat'].tail(30),
                         name='Prevision', line=dict(color='green', dash='dash')))
fig.add_trace(go.Scatter(
    x=pd.concat([forecast['ds'].tail(30), forecast['ds'].tail(30).iloc[::-1]]),
    y=pd.concat([forecast['yhat_upper'].tail(30), forecast['yhat_lower'].tail(30).iloc[::-1]]),
    fill='toself', fillcolor='rgba(0,200,0,0.1)', line=dict(color='rgba(0,0,0,0)'),
    name='Intervalle de confiance'
))
fig.update_layout(title=f'Prevision Prophet 30 jours — {ticker}', height=450)
fig.show()

## 6. Conclusion

| Modele | Forces | Faiblesses |
|--------|--------|------------|
| **Prophet** | Simple, saisonnalite | Moins precis sur le court terme |
| **XGBoost** | Tres precis avec features | Ne capture pas les tendances longues |
| **LSTM** | Apprend les sequences | Plus lent, besoin de plus de donnees |